# Gold Aggregation
Build business-oriented aggregated tables from the silver layer for analytics.

In [ ]:
# Gold: sales_by_product
# Total quantity, revenue, and order count per product

from pyspark.sql import functions as F

sales = spark.table("silver_sales_orders")
products = spark.table("silver_products")

sales_by_product = sales \
    .groupBy("ProductID") \
    .agg(
        F.sum("OrderQty").alias("TotalQuantity"),
        F.sum("LineTotal").alias("TotalRevenue"),
        F.countDistinct("SalesOrderID").alias("OrderCount")
    ) \
    .join(products, "ProductID", "left") \
    .select("ProductID", "ProductName", "CategoryName", "TotalQuantity", "TotalRevenue", "OrderCount")

sales_by_product.write.format("delta").mode("overwrite").saveAsTable("gold_sales_by_product")
print(f"gold_sales_by_product: {sales_by_product.count()} rows")

In [ ]:
# Gold: sales_by_geography
# Revenue, order count, customer count per city/state/country

customers = spark.table("silver_customers_with_address")

sales_with_geo = sales.join(customers, "CustomerID", "left")

sales_by_geography = sales_with_geo \
    .groupBy("CountryRegion", "StateProvince", "City") \
    .agg(
        F.sum("LineTotal").alias("TotalRevenue"),
        F.countDistinct("SalesOrderID").alias("OrderCount"),
        F.countDistinct("CustomerID").alias("CustomerCount")
    )

sales_by_geography.write.format("delta").mode("overwrite").saveAsTable("gold_sales_by_geography")
print(f"gold_sales_by_geography: {sales_by_geography.count()} rows")

In [ ]:
# Gold: product_performance
# Products ranked by revenue with percentiles

from pyspark.sql.window import Window

window = Window.orderBy(F.desc("TotalRevenue"))

product_performance = sales_by_product \
    .withColumn("RevenueRank", F.row_number().over(window)) \
    .withColumn("RevenuePercentile", F.percent_rank().over(window)) \
    .select("ProductID", "ProductName", "CategoryName", "TotalRevenue", "RevenueRank", "RevenuePercentile")

product_performance.write.format("delta").mode("overwrite").saveAsTable("gold_product_performance")
print(f"gold_product_performance: {product_performance.count()} rows")

In [ ]:
# Gold: sales_outliers
# Orders with TotalDue beyond 2 standard deviations from the mean

order_totals = sales \
    .groupBy("SalesOrderID") \
    .agg(F.first("TotalDue").alias("TotalDue"))

stats = order_totals.agg(
    F.mean("TotalDue").alias("MeanOrderValue"),
    F.stddev("TotalDue").alias("StdDev")
).collect()[0]

mean_val = stats["MeanOrderValue"]
std_val = stats["StdDev"]

sales_outliers = order_totals \
    .withColumn("MeanOrderValue", F.lit(mean_val)) \
    .withColumn("StdDev", F.lit(std_val)) \
    .withColumn("ZScore", (F.col("TotalDue") - F.lit(mean_val)) / F.lit(std_val)) \
    .filter(F.abs(F.col("ZScore")) > 2) \
    .select("SalesOrderID", "TotalDue", "MeanOrderValue", "StdDev", "ZScore")

sales_outliers.write.format("delta").mode("overwrite").saveAsTable("gold_sales_outliers")
print(f"gold_sales_outliers: {sales_outliers.count()} rows")

In [ ]:
# Verify gold tables
gold_tables = [t for t in spark.catalog.listTables() if t.name.startswith("gold_")]
print(f"Gold tables created: {len(gold_tables)}")
for t in gold_tables:
    count = spark.table(t.name).count()
    print(f"  {t.name}: {count} rows")